---
title: "Access Data - Download Links and Alternate Links in HDA" 
subtitle: "This notebook demonstrates how to discover assets using PySTAC, access data through HDA download links, alternative locations and native S3 assets"
author: "Author: template"
tags: [Template]
thumbnail: img/EUMETSAT_TEST.png 
license: MIT
copyright: "© 2026 EUMETSAT"
---

<!-- Optional: Add a JupyterHub launch link here. If you don’t have one, you can remove this block. -->

<div style="margin: 6px 0;">
  <a href="https://jupyter.central.data.destination-earth.eu/user-redirect/lab/tree/template1.ipynb" target="_blank" style="text-decoration: none;">
    <span class="launch">🚀 Launch in JupyterHub</span>
  </a>
</div>


# Access Data - Download Links and Alternate Links in HDA

This notebook demonstrates how to:
- Discover assets using PySTAC
- Access data through HDA download links
- Access data through alternative locations
- Access native S3 assets
- Understand when each option should be used

(template.ipynb-Contents)=
### Contents

In the intro below, explain what the notebook does, how it works at a high level, and anything a user should know before running it.

- **Objective:** What is the main goal of this notebook?
- **Data Sources:** Where the data comes from (links if applicable).
- **Methods:** Briefly outline the approach or algorithms used.
- **Prerequisites:** Any installations, credentials, or setup needed.
- **Expected Output:** What the user can expect at the end (plots, tables, files, etc.).

(template.ipynb-Prerequisites)=
## Prerequisites
This section was inspired by [this template](https://github.com/alan-turing-institute/the-turing-way/blob/master/book/templates/chapter-template/chapter-landing-page.md) of the wonderful [The Turing Way](https://the-turing-way.netlify.app) Jupyter Book.

## Imports

In [ ]:
from pystac_client import Client
import xarray as xr
import fsspec
from pprint import pprint

## Connect to HDA STAC

In [ ]:
STAC_URL = 'https://hda.data.destination-earth.eu/stac'

catalog = Client.open(STAC_URL)
catalog

## Search a dataset

In [ ]:
search = catalog.search(
    collections=['EO.UFZ.DAT.HYDROLAND'],
    limit=1
)

item = next(search.items())
print(item.id)

## Inspect assets

In [ ]:
for asset_name, asset in item.assets.items():
    print('=' * 80)
    print('Asset:', asset_name)
    print('Href :', asset.href)

## Inspect full asset metadata

In [ ]:
asset_name = list(item.assets.keys())[0]
asset = item.assets[asset_name]

pprint(asset.to_dict())

## Extract available access URLs

In [ ]:
def extract_links(asset):
    links = {'download': asset.href}

    alternate = asset.extra_fields.get('alternate', {})

    for key, value in alternate.items():
        if isinstance(value, dict) and 'href' in value:
            links[key] = value['href']

    return links

links = extract_links(asset)
links

## Access through the HDA Download Service

In [ ]:
access_token = '<ACCESS_TOKEN>'

headers = {
    'Authorization': f'Bearer {access_token}'
}

download_url = links['download']

ds_download = xr.open_dataset(
    download_url,
    engine='h5netcdf',
    chunks={},
    decode_times=False,
    storage_options={
        'client_kwargs': {
            'trust_env': False,
            'headers': headers
        }
    }
)

ds_download

### When to use download links

- Simple authenticated access
- Local analysis
- Downloading a few files
- Tutorials and examples

## Access through an alternative location

In [ ]:
location_url = links.get('location')
print(location_url)

In [ ]:
if location_url:
    fs = fsspec.filesystem('http')

    f = fs.open(location_url)

    ds_location = xr.open_dataset(f)

    ds_location

### When to use alternative locations

- Direct access to provider storage
- Potentially lower latency
- Interactive access to individual files

## Access through Native S3

In [ ]:
native_url = links.get('native')
print(native_url)

In [ ]:
if native_url:
    ds_native = xr.open_dataset(
        native_url,
        engine='h5netcdf',
        chunks='auto',
        storage_options={'anon': True}
    )

    ds_native

### Cloud-native processing example

In [ ]:
if native_url:
    first_var = list(ds_native.data_vars)[0]
    ds_native[first_var].mean().compute()

## Decision Guide

| Use case | Recommended access |
|-----------|-------------------|
| Download a few files | Download link |
| Simple authenticated access | Download link |
| Direct access to provider storage | Alternative location |
| Interactive inspection | Alternative location |
| Dask workflows | Native S3 |
| Processing many files | Native S3 |
| Cloud-native analytics | Native S3 |

## Summary
Conclude with a brief single paragraph summarizing at a high level the key pieces that were learned and how they tied to your objectives. Look to reiterate what the most important takeaways were.

## Resources and references
Finally, be rigorous in your citations and references as necessary. Give credit where credit is due. Also, feel free to link to relevant external material, further reading, documentation, etc. Then you're done! Give yourself a quick review, a high five, and send us a pull request. A few final notes:
 - `Kernel > Restart Kernel and Run All Cells...` to confirm that your notebook will cleanly run from start to finish
 - `Kernel > Restart Kernel and Clear All Outputs...` before committing your notebook, our machines will do the heavy lifting
 - Take credit! Provide author contact information if you'd like; if so, consider adding information here at the bottom of your notebook
 - Give credit! Attribute appropriate authorship for referenced code, information, images, etc.
 - Only include what you're legally allowed: **no copyright infringement or plagiarism**
 
Thank you for your contribution!